In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_regression_model(y_test, y_pred, show_plots=True):
    """
    Evaluates regression model performance and shows diagnostic plots.

    Parameters:
        y_test (np.ndarray): True target values.
        y_pred (np.ndarray): Predicted target values.
        show_plots (bool): If True, shows diagnostic plots.

    Returns:
        dict: Dictionary of evaluation metrics.
    """
    y_pred = np.array(y_pred).flatten()
    y_test = np.array(y_test).flatten()

    # Metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    print(f"✅ Mean Absolute Error (MAE): {mae:.4f}")
    print(f"✅ Mean Squared Error (MSE): {mse:.4f}")
    print(f"✅ Root Mean Squared Error (RMSE): {rmse:.4f}")
    print(f"✅ R² Score: {r2:.4f}")

    # Diagnostic plots
    if show_plots:
        fig, axs = plt.subplots(1, 3, figsize=(18, 5))

        # 1. Residual Plot
        residuals = y_test - y_pred
        axs[0].scatter(y_pred, residuals, alpha=0.6)
        axs[0].hlines(0, xmin=min(y_pred), xmax=max(y_pred), colors='r', linestyles='dashed')
        axs[0].set_title("Residual Plot")
        axs[0].set_xlabel("Predicted")
        axs[0].set_ylabel("Residuals")

        # 2. Predicted vs. Actual
        axs[1].scatter(y_test, y_pred, alpha=0.6)
        axs[1].plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 'r--')
        axs[1].set_title("Predicted vs. Actual")
        axs[1].set_xlabel("Actual")
        axs[1].set_ylabel("Predicted")

        # 3. Error Distribution
        axs[2].hist(residuals, bins=30, alpha=0.7)
        axs[2].set_title("Error Distribution")
        axs[2].set_xlabel("Prediction Error")
        axs[2].set_ylabel("Frequency")

        plt.tight_layout()
        plt.show()

    return {
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "r2": r2
    }


In [1]:
from scipy.stats import ttest_rel, wilcoxon
import numpy as np

def statistical_tests(main_model_scores, baseline_model_scores, verbose=True):
    """
    Perform paired t-test and Wilcoxon signed-rank test on two paired score arrays.

    Parameters:
    - main_model_scores (array-like): Scores from the main model.
    - baseline_model_scores (array-like): Scores from the baseline model.
    - verbose (bool): If True, print the results.

    Returns:
    - dict: Dictionary with t-test and Wilcoxon test results.
    """
    # Convert to numpy arrays and validate
    main_model_scores = np.asarray(main_model_scores)
    baseline_model_scores = np.asarray(baseline_model_scores)

    if main_model_scores.shape != baseline_model_scores.shape:
        raise ValueError("Input score arrays must have the same shape.")

    # Paired t-test (assumes scores are normally distributed)
    t_stat, p_value_ttest = ttest_rel(main_model_scores, baseline_model_scores)

    # Wilcoxon signed-rank test (non-parametric)
    try:
        w_stat, p_value_wilcoxon = wilcoxon(main_model_scores, baseline_model_scores)
    except ValueError as e:
        w_stat, p_value_wilcoxon = np.nan, np.nan
        if verbose:
            print(f"Wilcoxon test error: {e}")

    if verbose:
        print(f"Paired t-test:           t = {t_stat:.4f}, p = {p_value_ttest:.4f}")
        print(f"Wilcoxon signed-rank:   W = {w_stat:.4f}, p = {p_value_wilcoxon:.4f}")

    return {
        "ttest": {"t_stat": t_stat, "p_value": p_value_ttest},
        "wilcoxon": {"w_stat": w_stat, "p_value": p_value_wilcoxon}
    }

# ✅ 3. Interpret the Result
# If p < 0.05, you can claim statistical significance.

# 🎯 Example input: MAE scores from 5 test folds
main_model_scores = [0.5080,  0.1902,  0.2804, 0.2854, 0.2075]
baseline_model_scores = [0.5586, 0.4946,  0.5407, 0.5333, 0.5386]

# 🔍 Run the tests
results = statistical_tests(main_model_scores, baseline_model_scores)

Paired t-test:           t = -4.8362, p = 0.0084
Wilcoxon signed-rank:   W = 0.0000, p = 0.0625


In [3]:
import pandas as pd
import numpy as np

def generate_random_date():
    start_date = pd.to_datetime("2005-12-01", utc=True)
    end_date = pd.to_datetime("now", utc=True)
    date_range = pd.date_range(start=start_date, end=end_date, freq="H")
    
    # Select and convert explicitly to pandas.Timestamp
    timestamp = pd.Timestamp(np.random.choice(date_range)).tz_convert("UTC")
    
    return timestamp.strftime("%Y-%m-%d %H:%M:%S %Z")
